# Pauli Algebra 01: Core Conventions

This notebook introduces the symbolic Pauli backend in `quantum_simulation.pauli_algebra`. The goal is to make the module conventions explicit before building Hamiltonians or running dynamics.


## What you will learn

1. How `PauliString` and `PauliSum` represent operators without building dense matrices.
2. How site labels map to computational-basis bits.
3. Why `@` means commutator in this module.
4. How to build local and translated Pauli-chain terms.


In [1]:
from pathlib import Path
import importlib
import shutil
import subprocess
import sys
import types


def _ensure_ipython_display_if_missing():
    try:
        import IPython.display  # noqa: F401
        return
    except ModuleNotFoundError:
        pass

    ipython_module = types.ModuleType("IPython")
    display_module = types.ModuleType("IPython.display")
    display_module.display = lambda *args, **kwargs: None
    display_module.Math = lambda *args, **kwargs: None
    ipython_module.display = display_module
    ipython_module.get_ipython = lambda: None
    ipython_module.version_info = (0, 0)
    sys.modules.setdefault("IPython", ipython_module)
    sys.modules.setdefault("IPython.display", display_module)


def _import_pauli_algebra():
    _ensure_ipython_display_if_missing()
    return importlib.import_module("quantum_simulation.pauli_algebra")


try:
    pa = _import_pauli_algebra()
except ModuleNotFoundError as exc:
    if exc.name != "quantum_simulation":
        raise
    repo_dir = Path("quantum-simulation")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/ToelUl/quantum-simulation.git"], check=True)
    target = Path("quantum_simulation")
    if not target.exists():
        shutil.copytree(repo_dir / "quantum_simulation", target)
    pa = _import_pauli_algebra()

print("Loaded quantum_simulation.pauli_algebra from", Path(pa.__file__).parent)


Loaded quantum_simulation.pauli_algebra from /mnt/d/phd_research_projects/quantum-simulation-main/quantum_simulation/pauli_algebra


In [2]:
import numpy as np
import torch

from quantum_simulation.pauli_algebra import (
    PauliString,
    PauliSum,
    pauli_string_from_sites,
    global_pauli_op_chain,
    global_pauli_op_chain_list,
    multiply_pauli_operators,
)

np.set_printoptions(precision=3, suppress=True)
print("torch", torch.__version__)


torch 2.10.0+cu130


## Single Pauli-string algebra

A `PauliString` stores one Pauli word and a complex coefficient. Ordinary multiplication keeps all phases, while `@` is reserved for the commutator.


In [3]:
X = PauliString("X")
Y = PauliString("Y")
Z = PauliString("Z")

print("X * Y =", X * Y)
print("Y * X =", Y * X)
print("X @ Y = [X, Y] =", X @ Y)
print("{X, Y} =", X.anticommutator(Y))

assert (X * Y).label == "Z"
assert np.isclose((X * Y).coeff, 1j)
assert (X @ Y).label == "Z"
assert np.isclose((X @ Y).coeff, 2j)
assert np.isclose(X.anticommutator(Y).coeff, 0.0)


X * Y = (1j)*Z
Y * X = (-1j)*Z
X @ Y = [X, Y] = (2j)*Z
{X, Y} = (0j)*Z


## Site and bit convention

`label[j]` acts on site `j`. The same site `j` is stored in bit `j` of a computational-basis index, so site 0 is the least significant bit.


In [4]:
n = 3
z0 = pauli_string_from_sites("Z", [0], lattice_length=n)
z1 = pauli_string_from_sites("Z", [1], lattice_length=n)
z2 = pauli_string_from_sites("Z", [2], lattice_length=n)

basis_index = 0b101
ket = np.zeros(2**n, dtype=complex)
ket[basis_index] = 1.0

for op in [z0, z1, z2]:
    phase = np.vdot(ket, op.to_matrix() @ ket)
    print(f"{op.label} on |{basis_index:03b}> gives phase {phase.real:+.0f}")

assert np.isclose(np.vdot(ket, z0.to_matrix() @ ket), -1.0)
assert np.isclose(np.vdot(ket, z1.to_matrix() @ ket), +1.0)
assert np.isclose(np.vdot(ket, z2.to_matrix() @ ket), -1.0)


ZII on |101> gives phase -1
IZI on |101> gives phase +1
IIZ on |101> gives phase -1


## `PauliSum` combines repeated labels

A `PauliSum` is the standard container for Hamiltonians and observables. Calling `simplify()` merges duplicate labels and drops near-zero coefficients.


In [5]:
raw = PauliSum([
    PauliString("XXI", 0.25),
    PauliString("XXI", 0.75),
    PauliString("ZII", -0.5),
    PauliString("ZII", 0.5),
])

simplified = raw.simplify()
print("raw       =", raw)
print("simplified=", simplified)

assert len(simplified.terms) == 1
assert simplified.terms[0].label == "XXI"
assert np.isclose(simplified.terms[0].coeff, 1.0)


raw       = ((0.25+0j))*XXI + ((0.75+0j))*XXI + ((-0.5+0j))*ZII + ((0.5+0j))*ZII
simplified= XXI


## Local terms and translated chains

Use `pauli_string_from_sites` for a local operator, and `global_pauli_op_chain_list` or `global_pauli_op_chain` for all translated copies of a 1D Pauli word.


In [6]:
local_xz = pauli_string_from_sites("XZ", sites=[0, 2], lattice_length=4, coefficient=-0.5)
xx_terms_obc = global_pauli_op_chain_list("XX", lattice_length=4, coefficient=-1.0, pbc=False)
xx_chain_obc = global_pauli_op_chain("XX", lattice_length=4, coefficient=-1.0, pbc=False)

print("local X(site 0) Z(site 2):", local_xz)
print("open-boundary XX labels:", [term.label for term in xx_terms_obc])
print("summed XX chain:", xx_chain_obc)

assert local_xz.label == "XIZI"
assert [term.label for term in xx_terms_obc] == ["XXII", "IXXI", "IIXX"]
assert len(xx_chain_obc.terms) == 3


local X(site 0) Z(site 2): ((-0.5+0j))*XIZI
open-boundary XX labels: ['XXII', 'IXXI', 'IIXX']
summed XX chain: ((-1+0j))*XXII + ((-1+0j))*IXXI + ((-1+0j))*IIXX


## Ordinary products versus commutators

In this module, `A @ B` means `[A, B]`. If you need the ordinary product `AB`, use `multiply_pauli_operators` or `PauliString.multiply`.


In [7]:
A = pauli_string_from_sites("Z", [0], lattice_length=2)
B = pauli_string_from_sites("X", [0], lattice_length=2)

ordinary_product = multiply_pauli_operators(A, B)
commutator = PauliSum([A]) @ PauliSum([B])

print("A B       =", ordinary_product)
print("[A, B]    =", commutator)
print("dense [A,B] check:")
print(A.to_matrix() @ B.to_matrix() - B.to_matrix() @ A.to_matrix())

assert ordinary_product.terms[0].label == "YI"
assert np.isclose(ordinary_product.terms[0].coeff, 1j)
assert np.allclose(commutator.to_matrix(), A.to_matrix() @ B.to_matrix() - B.to_matrix() @ A.to_matrix())


A B       = (1j)*YI
[A, B]    = (2j)*YI
dense [A,B] check:
[[ 0.+0.j  2.+0.j  0.+0.j  0.+0.j]
 [-2.+0.j  0.+0.j  0.+0.j  0.+0.j]
 [ 0.+0.j  0.+0.j  0.+0.j  2.+0.j]
 [ 0.+0.j  0.+0.j -2.+0.j  0.+0.j]]


## Takeaway

For this backend, the safest workflow is: build symbolic Pauli terms, simplify them, use `@` only for commutators, and call `to_matrix()` only for small diagnostics or dense cross-checks.
